In [13]:
!pip install torch transformers datasets peft matplotlib scikit-learn

In [14]:
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import time
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

In [15]:
MODEL_NAME = "distilbert-base-uncased"
DATASET_NAME = "imdb"
TARGET_MODULES = ["q_lin", "v_lin"]
LORA_RANK = 8
MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 1

In [16]:
def load_and_prepare_data():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_fn(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

    dataset = load_dataset(DATASET_NAME)
    tokenized_data = dataset.map(tokenize_fn, batched=True)
    train_data = tokenized_data["train"].shuffle(seed=42).select(range(2000))
    eval_data = tokenized_data["test"].shuffle(seed=42).select(range(500))
    return train_data, eval_data

In [17]:
def get_full_ft_args():
    return TrainingArguments(
        output_dir="./full_ft_results",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        num_train_epochs=MAX_EPOCHS,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        weight_decay=0.01,
        report_to="none"
    )

In [19]:
import torch.nn.functional as F

def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions
    loss = F.cross_entropy(torch.tensor(logits), torch.tensor(labels)).item()
    accuracy = accuracy_score(labels, logits.argmax(-1))
    return {"accuracy": accuracy, "eval_loss": loss}

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_threshold=0.001
)

In [20]:
def run_full_finetuning(train_data, eval_data):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )
    args = get_full_ft_args()
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        compute_metrics=compute_metrics,
        callbacks=[early_stopping]
    )
    start_time = time.time()
    trainer.train()
    ft_time = time.time() - start_time
    eval_results = trainer.evaluate()
    return {
        "time": ft_time / 60,
        "accuracy": eval_results["eval_accuracy"],
        "memory": torch.cuda.max_memory_allocated() / (1024 ** 3),
        "epochs": trainer.state.epoch
    }

In [23]:
torch.cuda.empty_cache()
train_data, eval_data = load_and_prepare_data()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [24]:
print("Running Full Fine-Tuning...")
ft_results = run_full_finetuning(train_data, eval_data)

Running Full Fine-Tuning...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy
1,0.312100,0.297758,0.870000
2,0.261700,0.356178,0.882000


In [29]:
for metric in ["time", "accuracy"]:
  print(f"{metric:<15} | {ft_results[metric]:<10.2f}")

time            | 103.80    
accuracy        | 0.87      
